In [38]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [39]:
word_index = imdb.get_word_index()
reverse_word_index = {value:key for key, value in word_index.items()}

In [40]:
model =load_model('simple_rnn_imdb_model.h5')
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [41]:
model.get_weights()

[array([[ 0.07016116, -0.0366113 , -0.005948  , ...,  0.03381253,
          0.01921761, -0.04595463],
        [ 0.06123089, -0.03321686,  0.02336419, ...,  0.00261296,
          0.02532764, -0.01998516],
        [ 0.00606986, -0.03634225, -0.03319365, ...,  0.0125521 ,
          0.0148513 ,  0.04050291],
        ...,
        [ 0.02728435, -0.00473743,  0.05807196, ..., -0.01815178,
         -0.02754336, -0.03475129],
        [-0.01572731,  0.02041738, -0.02715932, ...,  0.00618465,
          0.00687384,  0.06064425],
        [ 0.05558305, -0.00503255, -0.07220824, ..., -0.06082191,
          0.0186367 ,  0.08519207]], dtype=float32),
 array([[ 0.14624639,  0.08794306,  0.09818003, ...,  0.03779534,
         -0.05210674,  0.04075695],
        [-0.00021307, -0.04007296, -0.20764987, ...,  0.05050532,
          0.15302059, -0.09452144],
        [-0.09544288,  0.11451583, -0.15764798, ..., -0.12453514,
         -0.0788773 , -0.14731498],
        ...,
        [-0.0155498 , -0.0479637 ,  0.0

In [42]:
# Step2: Helper Functions
# Function to decide reviews
def decode_review(encoded_review):
    return " ".join([reverse_word_index.get(i-3,"?") for i in encoded_review])
import re
import numpy as np
from tensorflow.keras.preprocessing import sequence
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    words = text.split()
    encoded_review = []
    for word in words:
        index = word_index.get(word)
        if index is None:
            index = 2
        else:
            index = index + 3
            if index >= 10000:
                index = 2
        encoded_review.append(index)
    padded_review = sequence.pad_sequences(
        [encoded_review],
        maxlen=500
    )
    return padded_review
def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)
    prediction = model.predict(
        preprocessed_input,
        verbose=0
    )
    score = float(prediction[0][0])
    sentiment = "Positive" if score >= 0.5 else "Negative"
    return sentiment, score

In [45]:
example_review = "This movie was trash and not worth anyone's time"
sentiment, score = predict_sentiment(example_review)
print(f"Review: {example_review}")
print(f"Sentiment: {sentiment}")
print(f"Prediction Score: {score:.4f}")

Review: This movie was trash and not worth anyone's time
Sentiment: Negative
Prediction Score: 0.3965
